In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt

In [3]:
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)

stk_data = yf.download("RELIANCE.NS", start=start, end=end)

[*********************100%***********************]  1 of 1 completed


In [4]:
stk_data

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2013-06-03,166.953690,170.651287,166.392150,170.301652,14165128
2013-06-04,165.322067,169.475258,165.099582,167.525804,14752690
2013-06-05,169.941437,170.428800,165.385639,165.385639,12748842
2013-06-06,167.822449,170.344022,166.996060,169.517633,17113393
2013-06-07,166.042526,169.941434,165.491595,168.288627,9420701
...,...,...,...,...,...
2022-02-04,1060.769531,1068.573059,1056.128431,1065.183162,11061241
2022-02-07,1054.308350,1072.372316,1048.802625,1065.638136,10714467


In [5]:
# The dataframe is not normal columns, it’s a MultiIndex (two-level columns).
# So, flattening the columns:

stk_data.columns = stk_data.columns.get_level_values(0)

In [6]:
stk_data=stk_data[["Open", "High", "Low", "Close"]]

In [7]:
stk_data

Price,Open,High,Low,Close
Date,,,,
2013-06-03,170.301652,170.651287,166.392150,166.953690
2013-06-04,167.525804,169.475258,165.099582,165.322067
2013-06-05,165.385639,170.428800,165.385639,169.941437
2013-06-06,169.517633,170.344022,166.996060,167.822449
2013-06-07,168.288627,169.941434,165.491595,166.042526
...,...,...,...,...
2022-02-04,1065.183162,1068.573059,1056.128431,1060.769531
2022-02-07,1065.638136,1072.372316,1048.802625,1054.308350
2022-02-08,1060.473871,1073.828498,1051.077815,1072.031128


In [8]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1 = Ms.fit_transform(stk_data)
print("Len:", data1.shape)

Len: (2144, 4)


In [9]:
data1 = pd.DataFrame(data1, columns=["Open", "High", "Low", "Close"])

### Train - Test Split

In [10]:
training_size = round(len(data1) * 0.80)
print(training_size)

X_train = data1[:training_size]
X_test = data1[training_size:]
print("X_train length:", X_train.shape)
print("X_test length:", X_test.shape)

y_train = data1[:training_size]
y_test = data1[training_size:]
print("y_train length:", y_train.shape)
print("y_test length:", y_test.shape)

1715
X_train length: (1715, 4)
X_test length: (429, 4)
y_train length: (1715, 4)
y_test length: (429, 4)


### Model Creation

#### VAR Model

In [11]:
performance = {"Model": [], "RMSE": [], "MaPe": [], "Lag": [], "Test": []}

In [12]:
def combination(dataset, list1):
    print(list1)   # show which variables we are using (e.g., Close, High)

    datasetTwo = dataset[list1]  # select only chosen columns (multivariate input for VAR)

    test_obs = 28                      # number of future days to test (like 1-month prediction)
    train = datasetTwo[:-test_obs]     # training data (past data)
    test = datasetTwo[-test_obs:]      # testing data (last 28 days to compare)

    from statsmodels.tsa.api import VAR  # import VAR model

    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)           # create VAR model
        results = model.fit(i)       # fit model with lag = i


        # check model quality (lower is better)
        print("Order =", i)          # Shows how many lag values the model is using (past days used for prediction)
        print("AIC :", results.aic)  # AIC (Akaike Information Criterion) score tells how good the model is (lower = better)
        print("BIC :", results.bic)  # BIC (Bayesian Information Criterion) score also measures model quality but penalizes complexity more (lower=better)     
        

    x = model.select_order(maxlags=12)  # automatically find best lag
    order = x.selected_orders['aic']    # select best lag using AIC

    result = model.fit(order)   # fit model with best lag

    lagged_values = train.values[-order:]  # take last 'lag' values for prediction input
    pred = result.forecast(y=lagged_values, steps=28)  # forecast next 28 days

    import pandas as pd
    preds = pd.DataFrame(pred, columns=list1)   # convert predictions to DataFrame

    # Evaluation
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

    mse = mean_squared_error(test, pred)  # calculate error between actual and predicted
    rmse = round(np.sqrt(mse), 3)         # convert MSE to RMSE (actual error scale)

    mape = mean_absolute_percentage_error(test, pred)  # percentage error (easy to interpret)

    # Store results
    performance["Model"].append(list1)      # store variable combination used
    performance["RMSE"].append(rmse)        # store RMSE
    performance["MaPe"].append(mape)        # store MAPE
    performance["Lag"].append(order)        # store lag
    performance["Test"].append(test_obs)    # store test size

    perf = pd.DataFrame(performance)        # convert results to table

    return perf, result, pred               # return summary + model + predictions

In [13]:
combination(data1, ["Close", "High"])
combination(data1, ["Close", "High", "Open"])
combination(data1, ["Close", "High", "Open", "Low"])

['Close', 'High']
Order = 1
AIC : -20.47004717605133
BIC : -20.453999487848552
Order = 2
AIC : -20.50752274069618
BIC : -20.480766178887144
Order = 3
AIC : -20.54177252142764
BIC : -20.504298741850437
Order = 4
AIC : -20.556832112161615
BIC : -20.508632760075027
Order = 5
AIC : -20.55810024935467
BIC : -20.499166959419725
Order = 6
AIC : -20.56026482821715
BIC : -20.490589224478654
Order = 7
AIC : -20.561665759272405
BIC : -20.481239455140418
Order = 8
AIC : -20.577685630213907
BIC : -20.486500228445177
Order = 9
AIC : -20.58363182815029
BIC : -20.481678920829662
Order = 10
AIC : -20.583043938482348
BIC : -20.47031510700412
['Close', 'High', 'Open']
Order = 1
AIC : -31.849598179085884
BIC : -31.81750280268032
Order = 2
AIC : -31.903378533156015
BIC : -31.84718975335704
Order = 3
AIC : -31.9356534179237
BIC : -31.855352461686838
Order = 4
AIC : -31.947143956810685
BIC : -31.842712027289746
Order = 5
AIC : -31.95483237954918
BIC : -31.82625065605476
Order = 6
AIC : -31.959949839772033
BI

(                      Model   RMSE      MaPe  Lag  Test
 0             [Close, High]  0.039  0.034125   11    28
 1       [Close, High, Open]  0.039  0.034335   12    28
 2  [Close, High, Open, Low]  0.036  0.032014   12    28,
 array([[0.84717649, 0.85032239, 0.84441386, 0.84673677],
        [0.84940639, 0.85440185, 0.84696974, 0.84762651],
        [0.85053399, 0.85428654, 0.84658381, 0.84922714],
        [0.84604134, 0.85089207, 0.84779083, 0.84732207],
        [0.84798664, 0.84981857, 0.84435783, 0.84708635],
        [0.84963263, 0.85279699, 0.84484122, 0.84846657],
        [0.8534902 , 0.85566206, 0.84838929, 0.85230823],
        [0.8513496 , 0.85495428, 0.85005344, 0.85170962],
        [0.85059935, 0.85430528, 0.84867819, 0.85052407],
        [0.8513048 , 0.85645816, 0.85015138, 0.85024027],
        [0.85090543, 0.85543067, 0.84910871, 0.84975233],
        [0.85096118, 0.8554047 , 0.84927448, 0.85009486],
        [0.85087422, 0.85561143, 0.8488818 , 0.84967097],
        [0.850796

In [14]:
perf = pd.DataFrame(performance)
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High]",0.039,0.034125,11,28
1,"[Close, High, Open]",0.039,0.034335,12,28
2,"[Close, High, Open, Low]",0.036,0.032014,12,28


<h5 style="color:darkblue;"> 
• Best model: [Close, High, Open, Low]<br><br>  
    Because, <br>
    Lowest RMSE → more accurate<br>
    Lowest MAPE → smallest % error<br><br> 

• We tested different combinations of stock variables using VAR and found that using all variables (Open, High, Low, Close) gives the best prediction with lowest error.
</h5>